Nombre: Maria Jose Gonzalez Serrano
Asignatura: Fundamentos de Programación en Python
Profesor: David Andrés Franco Quintero

Examen Final : Cajero Electrónico

El proyecto se enfoca en desarrollar un cajero automático utilizando Python y archivos JSON para gestionar la información de las cuentas.

El programa comenzará pidiendo al usuario que ingrese su número de cuenta y contraseña, permitiendo hasta tres intentos. Si la cuenta no existe o la contraseña es incorrecta, el sistema mostrará un mensaje apropiado.

Una vez que el usuario ingrese correctamente, podrá acceder a un menú con opciones para consultar el saldo, retirar dinero, ahorrar dinero y salir.

En la opción de retirar dinero, el sistema verificará que haya suficiente saldo y que haya billetes disponibles en el cajero. Para la opción de ahorrar dinero, el usuario ingresará un monto y el sistema comprobará si el dinero depositado es menor, igual o mayor al valor esperado.

Finalmente, al salir, el programa guardará todos los cambios realizados en los archivos JSON y cerrará el sistema de manera adecuada. Además, se implementarán validaciones para evitar la entrada de letras o caracteres no válidos en los datos numéricos.

In [8]:

# Examen final - Cajero
# Estudiante: Maria Jose Gonzalez Serrano
# Profe: David Franco

import json


def abrir_cuentas():
    archivo = open('cuenta-clave.json', 'r')
    datos = json.load(archivo)
    archivo.close()
    return datos

def abrir_billetes():
    archivo = open('denominacion-billetes.json', 'r')
    datos = json.load(archivo)
    archivo.close()
    return datos

def guardar_cuentas(cuentas):
    archivo = open('cuenta-clave.json', 'w')
    json.dump(cuentas, archivo)
    archivo.close()

def guardar_billetes(billetes):
    archivo = open('denominacion-billetes.json', 'w')
    json.dump(billetes, archivo)
    archivo.close()

def revisar_numero(texto):
    if texto == "":
        return False
    for letra in texto:
        if letra < '0' or letra > '9':
            return False
    return True

def buscar_cuenta(numero, lista):
    for c in lista:
        if str(c["CUENTA"]) == numero:
            return c
    return None

class Cajero:

    def __init__(self):
        self.cuentas = abrir_cuentas()
        self.billetes = abrir_billetes()
        self.usuario = None
        self.saldo = 0

    def inicio(self):
        intentos = 3

        while intentos > 0:
            print("=" * 50)
            print("        CAJERO ELECTRONICO")
            print("=" * 50)

            print("")
            cuenta_ingresada = input("Numero de cuenta (10 digitos): ")
            if len(cuenta_ingresada) != 10:
                print("")
                print("*** La cuenta debe tener 10 digitos ***")
                intentos = intentos - 1
                print("Te quedan:", intentos, "intentos")
                continue

            if revisar_numero(cuenta_ingresada) == False:
                print("")
                print("*** No se hallo su cuenta ***")
                intentos = intentos - 1
                print("Te quedan:", intentos, "intentos")
                continue

            self.usuario = buscar_cuenta(cuenta_ingresada, self.cuentas)

            if self.usuario == None:
                print("")
                print("*** No se hallo su cuenta ***")
                intentos = intentos - 1
                print("Te quedan:", intentos, "intentos")
                continue


            while intentos > 0:
                clave = input("Contraseña (4 digitos): ")

                if str(self.usuario["CONTRASENA"]) == clave:
                    self.saldo = self.usuario["SALDO"]
                    print("")
                    print("INGRESO EXITOSO")
                    return True
                else:
                    print("")
                    print("Contraseña invalida")
                    intentos = intentos - 1
                    if intentos > 0:
                        print("Te quedan:", intentos, "intentos")
                    else:
                        break

        print("")
        print("=" * 50)
        print("GASTO TODOS SUS INTENTOS")
        print("EL PROGRAMA SE CERRARA")
        print("=" * 50)
        return False

    def ver_saldo(self):
        print("")
        print("-" * 40)
        print("  SU SALDO ES: $", self.saldo)
        print("-" * 40)

        while True:
            print("")
            print("QUE DESEA HACER?")
            print("1. Retirar")
            print("2. Ahorrar")
            print("3. Salir")

            op = input("Opcion: ")

            if op == "1":
                self.retirar()
                return
            elif op == "2":
                self.ahorrar()
                return
            elif op == "3":
                return
            else:
                print("Opcion no valida")


    def hay_billetes(self, monto):
        copia = []
        for b in self.billetes:
            copia.append({
                "DENOMINACION": b["DENOMINACION"],
                "BILLETES": b["BILLETES"]
            })

        resto = monto

        # billetes de 100
        for b in copia:
            if b["DENOMINACION"] == 100000:
                if b["BILLETES"] > 0:
                    usar = resto // 100000
                    if usar > b["BILLETES"]:
                        usar = b["BILLETES"]
                    resto = resto - (usar * 100000)
                    b["BILLETES"] = b["BILLETES"] - usar
                break

        # billetes de 50
        for b in copia:
            if b["DENOMINACION"] == 50000:
                if b["BILLETES"] > 0:
                    usar = resto // 50000
                    if usar > b["BILLETES"]:
                        usar = b["BILLETES"]
                    resto = resto - (usar * 50000)
                    b["BILLETES"] = b["BILLETES"] - usar
                break

        # billetes de 20
        for b in copia:
            if b["DENOMINACION"] == 20000:
                if b["BILLETES"] > 0:
                    usar = resto // 20000
                    if usar > b["BILLETES"]:
                        usar = b["BILLETES"]
                    resto = resto - (usar * 20000)
                    b["BILLETES"] = b["BILLETES"] - usar
                break

        # billetes de 10
        for b in copia:
            if b["DENOMINACION"] == 10000:
                if b["BILLETES"] > 0:
                    usar = resto // 10000
                    if usar > b["BILLETES"]:
                        usar = b["BILLETES"]
                    resto = resto - (usar * 10000)
                    b["BILLETES"] = b["BILLETES"] - usar
                break

        if resto == 0:
            return True
        else:
            return False

    def calcular_billetes(self, monto):
        entregar = []
        resto = monto
        copia = []
        for b in self.billetes:
            copia.append({
                "DENOMINACION": b["DENOMINACION"],
                "BILLETES": b["BILLETES"]
            })

        # 100
        for b in copia:
            if b["DENOMINACION"] == 100000:
                if b["BILLETES"] > 0 and resto >= 100000:
                    usar = resto // 100000
                    if usar > b["BILLETES"]:
                        usar = b["BILLETES"]
                    if usar > 0:
                        entregar.append({"DENOMINACION": 100000, "CANTIDAD": usar})
                        resto = resto - (usar * 100000)
                        b["BILLETES"] = b["BILLETES"] - usar
                break

        # 50
        for b in copia:
            if b["DENOMINACION"] == 50000:
                if b["BILLETES"] > 0 and resto >= 50000:
                    usar = resto // 50000
                    if usar > b["BILLETES"]:
                        usar = b["BILLETES"]
                    if usar > 0:
                        entregar.append({"DENOMINACION": 50000, "CANTIDAD": usar})
                        resto = resto - (usar * 50000)
                        b["BILLETES"] = b["BILLETES"] - usar
                break

        # 20
        for b in copia:
            if b["DENOMINACION"] == 20000:
                if b["BILLETES"] > 0 and resto >= 20000:
                    usar = resto // 20000
                    if usar > b["BILLETES"]:
                        usar = b["BILLETES"]
                    if usar > 0:
                        entregar.append({"DENOMINACION": 20000, "CANTIDAD": usar})
                        resto = resto - (usar * 20000)
                        b["BILLETES"] = b["BILLETES"] - usar
                break

        # 10
        for b in copia:
            if b["DENOMINACION"] == 10000:
                if b["BILLETES"] > 0 and resto >= 10000:
                    usar = resto // 10000
                    if usar > b["BILLETES"]:
                        usar = b["BILLETES"]
                    if usar > 0:
                        entregar.append({"DENOMINACION": 10000, "CANTIDAD": usar})
                        resto = resto - (usar * 10000)
                        b["BILLETES"] = b["BILLETES"] - usar
                break

        return entregar, copia

    def retirar(self):
        if self.saldo < 10000:
            print("")
            print("=" * 50)
            print(" SU SALDO ES MENOR A 10000 ")
            print(" NO PUEDE RETIRAR DINERO")
            print("=" * 50)
            input("Presione Enter...")
            return

        while True:
            print("")
            print("-" * 40)
            print("        RETIRO DE DINERO")
            print("-" * 40)
            print("1. 40.000 ")
            print("2. 50.000 ")
            print("3. 100.000 ")
            print("4. Otro valor")
            print("5. Salir")

            op = input("Seleccione: ")

            if op == "5":
                return

            monto = 0

            if op == "1":
                monto = 40000
            elif op == "2":
                monto = 50000
            elif op == "3":
                monto = 100000
            elif op == "4":
                print("")
                print("Ingrese el monto (multiplo de 10000):")
                valor = input("Monto: ")

                if revisar_numero(valor) == False:
                    print("*** Eso no es un numero ***")
                    continue

                monto = int(valor)

                if monto % 10000 != 0:
                    print("Tiene que ser multiplo de 10000")
                    continue
            else:
                print("Opcion no valida")
                continue

            if self.saldo < monto:
                print("")
                print("NO TIENE FONDOS EN LA CUENTA")
                continue

            if self.hay_billetes(monto) == False:
                print("")
                print("EL CAJERO NO TIENE EFECTIVO")
                continue

            combinacion, nuevos = self.calcular_billetes(monto)

            self.billetes = nuevos
            self.saldo = self.saldo - monto

            print("")
            print("=" * 50)
            print("        RETIRO EXITOSO")
            print("=" * 50)
            print("Retiraste: $", monto)
            print("")
            print("Billetes que te doy:")

            for b in combinacion:
                cant = b["CANTIDAD"]
                denom = b["DENOMINACION"]
                print("  $", denom, "x", cant, "= $", denom * cant)

            print("")
            print("Nuevo saldo: $", self.saldo, "COP")
            print("=" * 50)

            input("Presione Enter")
            return

    def ahorrar(self):
        print("")
        print("=" * 50)
        print("        AHORRAR DINERO")
        print("=" * 50)

        # pido el monto
        while True:
            print("")
            print("Cuanto quieres ahorrar?")
            entrada = input("Monto: ")

            if revisar_numero(entrada) == False:
                print("Pon un numero")
                continue

            monto_pretendido = int(entrada)
            break

        print("")
        print("-" * 40)
        print("Cuantos billetes metes?")
        print("-" * 40)

        billetes_ingresados = []

        # 2000
        while True:
            print("Billetes de $2000:")
            cant = input("Cantidad: ")
            if revisar_numero(cant):
                cant = int(cant)
                if cant >= 0:
                    billetes_ingresados.append({"DENOMINACION": 2000, "CANTIDAD": cant})
                    break
            print("Numero no valido")

        # 5000
        while True:
            print("Billetes de $5000:")
            cant = input("Cantidad: ")
            if revisar_numero(cant):
                cant = int(cant)
                if cant >= 0:
                    billetes_ingresados.append({"DENOMINACION": 5000, "CANTIDAD": cant})
                    break
            print("Numero no valido")

        # 10000
        while True:
            print("Billetes de $10000:")
            cant = input("Cantidad: ")
            if revisar_numero(cant):
                cant = int(cant)
                if cant >= 0:
                    billetes_ingresados.append({"DENOMINACION": 10000, "CANTIDAD": cant})
                    break
            print("Numero no valido")

        # 20000
        while True:
            print("Billetes de $20000:")
            cant = input("Cantidad: ")
            if revisar_numero(cant):
                cant = int(cant)
                if cant >= 0:
                    billetes_ingresados.append({"DENOMINACION": 20000, "CANTIDAD": cant})
                    break
            print("Numero no valido")

        # 50000
        while True:
            print("Billetes de $50000:")
            cant = input("Cantidad: ")
            if revisar_numero(cant):
                cant = int(cant)
                if cant >= 0:
                    billetes_ingresados.append({"DENOMINACION": 50000, "CANTIDAD": cant})
                    break
            print("Numero no valido")

        # 100000
        while True:
            print("Billetes de $100000:")
            cant = input("Cantidad: ")
            if revisar_numero(cant):
                cant = int(cant)
                if cant >= 0:
                    billetes_ingresados.append({"DENOMINACION": 100000, "CANTIDAD": cant})
                    break
            print("Numero no valido")

        total = 0
        for b in billetes_ingresados:
            total = total + (b["DENOMINACION"] * b["CANTIDAD"])

        print("")
        print("Total que metiste: $", total, "COP")
        print("Querias ahorrar: $", monto_pretendido, "COP")


        if total < monto_pretendido:
            print("")
            print("No alcanza lo que querias ahorrar ")
            input("Presione Enter")
            return

        # actualizar billetes del cajero (solo de 10000 para arriba)
        # los de 2000 y 5000 no porque el cajero no da esos billetes
        for b_user in billetes_ingresados:
            if b_user["DENOMINACION"] >= 10000 and b_user["CANTIDAD"] > 0:
                for b_cajero in self.billetes:
                    if b_cajero["DENOMINACION"] == b_user["DENOMINACION"]:
                        b_cajero["BILLETES"] = b_cajero["BILLETES"] + b_user["CANTIDAD"]


        if total == monto_pretendido:
            self.saldo = self.saldo + total
            print("")
            print("=" * 50)
            print("        AHORRO EXITOSO")
            print("=" * 50)
            print("Ahorraste $", total, "COP")
            print("Tu saldo ahora: $", self.saldo, "COP")
            print("=" * 50)
            input("Presione Enter...")
            return


        excedente = total - monto_pretendido
        print("")
        print("TE SOBRARON $", excedente)

        while True:
            print("")
            print("Que hago con lo que sobra?")
            print("1. Donar a caridad")
            print("2. Mandar a otra cuenta")

            op_ex = input("Elige: ")

            if op_ex == "1":
                print("")
                print("=" * 50)
                print("        GRACIAS POR TU SOLIDARIDAD")
                print("=" * 50)
                self.saldo = self.saldo + monto_pretendido
                break

            elif op_ex == "2":
                while True:
                    print("")
                    print("Cuenta destino:")
                    cuenta_destino = input("Numero: ")

                    # validar que sean 10 digitos
                    if len(cuenta_destino) != 10:
                        print("La cuenta debe tener 10 digitos")
                        continue

                    if revisar_numero(cuenta_destino) == False:
                        print("La cuenta debe ser numerica")
                        continue

                    cuenta_valida = buscar_cuenta(cuenta_destino, self.cuentas)

                    if cuenta_valida == None:
                        print("Esa cuenta no existe")
                    else:
                        cuenta_valida["SALDO"] = cuenta_valida["SALDO"] + excedente
                        self.saldo = self.saldo + monto_pretendido
                        print("")
                        print("Transferido a cuenta", cuenta_destino)
                        break
                break

            else:
                print("Opcion no valida")

        print("")
        print("=" * 50)
        print("        AHORRO EXITOSO")
        print("=" * 50)
        print("Ahorraste $", monto_pretendido)
        print("Tu saldo: $", self.saldo)
        print("=" * 50)
        input("Presione Enter")

    def salir_sistema(self):
        for c in self.cuentas:
            if c["CUENTA"] == self.usuario["CUENTA"]:
                c["SALDO"] = self.saldo
                break

        guardar_cuentas(self.cuentas)
        guardar_billetes(self.billetes)

        print("")
        print("=" * 50)
        print(" Saliste del cajero ")
        print(" Hasta pronto ")
        print("=" * 50)

    def menu(self):
        while True:
            print("")
            print("=" * 50)
            print("            MENU")
            print("=" * 50)
            print("1. Ver saldo")
            print("2. Retirar")
            print("3. Ahorrar")
            print("4. Salir")

            op = input("Opcion: ")

            if op == "1":
                self.ver_saldo()
            elif op == "2":
                self.retirar()
            elif op == "3":
                self.ahorrar()
            elif op == "4":
                self.salir_sistema()
                break
            else:
                print(" Opcion invalida ")

    def correr(self):
        if self.inicio():
            self.menu()



cajero = Cajero()
cajero.correr()

        CAJERO ELECTRONICO

Numero de cuenta (10 digitos): 6710438261
Contraseña (4 digitos): d

Contraseña invalida
Te quedan: 2 intentos
Contraseña (4 digitos): 1414

Contraseña invalida
Te quedan: 1 intentos
Contraseña (4 digitos): 8757

Contraseña invalida

GASTO TODOS SUS INTENTOS
EL PROGRAMA SE CERRARA
